# Gold quality: telemetry exceptions and reporting impact

Read the saved snapshot after Gold activity succeeds, explain vehicle-day exceptions, and reconcile record outcomes.

## 1. Set up and read the saved snapshot

Read the completed upstream tables so quality reporting uses the same saved snapshot as activity reporting.

In [ ]:
# Telemetry exceptions and reporting impact
# Read saved tables after Gold activity succeeds.
from pathlib import Path
from datetime import date

from pyspark.sql import functions as F

schema_name = "ev_telematics.fleet_activity_demo_20260910"
project_dir = Path("/Volumes/ev_telematics/fleet_activity_demo_20260910/files")
output_dir = project_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
start_date = date(2026, 8, 1)
end_date = date(2026, 8, 30)

# This notebook uses the Databricks-provided Spark session.
spark.conf.set("spark.sql.session.timeZone", "UTC")
print("Report period:", start_date, "to", end_date)

bronze = spark.table(schema_name + ".bronze_telemetry")
accepted = spark.table(schema_name + ".silver_telemetry")
quarantine = spark.table(schema_name + ".quarantine_telemetry")
day_issues = spark.table(schema_name + ".silver_day_issues")
gold = spark.table(schema_name + ".gold_daily_vehicle_activity")

## 2. Check the Gold vehicle-day domain

Validate the reporting grain and allowed statuses before attaching quality information.

### Learning note: `Column.isin`
`F.col("distance_status").isin("trusted", "untrusted", "missing_telemetry", "insufficient_readings")` checks whether a value belongs to the allowed list. `~` reverses that check to find unsupported values. Null values need the separate `isNull()` check because they do not become true under `~isin(...)`. This method is new to the active 02/03 notebook pattern; it is also used below for allowed Silver reasons and quarantine scopes.

In [ ]:
key_columns = ["vehicle_id", "event_date"]
day_columns = ["vehicle_id", "event_date", "depot_id", "depot_name", "model",
               "registered_vehicle", "distance_status", "reading_count"]
day_domain = gold.select(*day_columns)
if day_domain.groupBy(*key_columns).count().filter(F.col("count") > 1).count() != 0:
    raise ValueError("Duplicate Gold vehicle-day key")
if day_domain.filter(F.col("vehicle_id").isNull() | (F.trim("vehicle_id") == "") |
                     F.col("event_date").isNull()).count() != 0:
    raise ValueError("Null or blank Gold vehicle-day key")
if day_domain.filter(F.col("distance_status").isNull() | ~F.col("distance_status").isin(
        "trusted", "untrusted", "missing_telemetry", "insufficient_readings")).count() != 0:
    raise ValueError("Unknown distance status")

## 3. Count accepted readings by day

Aggregate accepted records before joining to preserve one row per vehicle-day.

In [ ]:
accepted_counts = accepted.groupBy(*key_columns).agg(F.count("*").alias("accepted_reading_count"))

## 4. Count rejected and held records separately

Keep record rejections separate from day-level holds so each quarantine outcome remains interpretable.

In [ ]:
rejected = quarantine.filter(F.col("quarantine_scope") == "record")
held = quarantine.filter(F.col("quarantine_scope") == "day_hold")
rejected_counts = rejected.groupBy(*key_columns).agg(F.count("*").alias("rejected_record_count"))
held_counts = held.groupBy(*key_columns).agg(F.count("*").alias("held_reading_count"))

## 5. Join counts onto the day hub

Retain every reporting day while attaching independent counts, and preview days with rejected or held records.

In [ ]:
quality = (day_domain.join(accepted_counts, key_columns, "left")
    .join(rejected_counts, key_columns, "left")
    .join(held_counts, key_columns, "left")
    .withColumn("accepted_reading_count", F.coalesce("accepted_reading_count", F.lit(0)))
    .withColumn("rejected_record_count", F.coalesce("rejected_record_count", F.lit(0)))
    .withColumn("held_reading_count", F.coalesce("held_reading_count", F.lit(0))))
(quality.filter((F.col("rejected_record_count") > 0) | (F.col("held_reading_count") > 0))
    .orderBy("event_date", "vehicle_id").show(8, truncate=False))

## 6. Define issue labels and priorities

Use consistent labels and a fixed priority order to make exception summaries comparable.

In [ ]:
issue_dictionary = spark.createDataFrame([
    ("conflicting_timestamp", "Conflicting timestamp", 10),
    ("odometer_decreased", "Odometer decreased", 20),
    ("rejected_reading_in_day", "Rejected reading in day", 30),
    ("missing_telemetry", "Missing telemetry", 40),
    ("insufficient_readings", "Insufficient readings", 50),
    ("unregistered_vehicle", "Vehicle absent from roster", 60),
], "issue_code string, issue_label string, primary_priority long")

## 7. Expand and deduplicate Silver reasons

Keep one occurrence of each reason per vehicle-day so repeated reasons do not inflate issue counts.

### Learning note: `F.explode`
`F.explode("reasons")` turns each element of a day's reasons array into a separate row. A day with two reasons therefore becomes two issue rows. `dropDuplicates()` then keeps one row per vehicle/date/reason, including when a reason is repeated in the array or source rows. Empty or null arrays produce no rows; the later check rejects an untrusted day without a Silver reason. This counts issue occurrences, **never held readings**: held readings were counted independently from quarantine rows in the rejected/held counts step.

In [ ]:
silver_issues = (day_issues.select(*key_columns, F.explode("reasons").alias("issue_code"))
    .dropDuplicates())

## 8. Add status and registry issues

Include missing, insufficient and unregistered days without changing the existing distance status.

In [ ]:
missing_issues = (day_domain.filter(F.col("distance_status") == "missing_telemetry")
    .select(*key_columns).withColumn("issue_code", F.lit("missing_telemetry")))
insufficient_issues = (day_domain.filter(F.col("distance_status") == "insufficient_readings")
    .select(*key_columns).withColumn("issue_code", F.lit("insufficient_readings")))
registry_issues = (day_domain.filter(~F.col("registered_vehicle"))
    .select(*key_columns).withColumn("issue_code", F.lit("unregistered_vehicle")))
issue_rows = (silver_issues.unionByName(missing_issues)
    .unionByName(insufficient_issues).unionByName(registry_issues))

## 9. Join the dictionary and validate issues

Reject unknown reasons and orphan issues before they can enter reporting outputs.

In [ ]:
issues = issue_rows.join(issue_dictionary, "issue_code", "left")
if issues.filter(F.col("primary_priority").isNull()).count() != 0:
    raise ValueError("Unknown issue reason")
if silver_issues.filter(~F.col("issue_code").isin(
        "conflicting_timestamp", "odometer_decreased", "rejected_reading_in_day")).count() != 0:
    raise ValueError("Unsupported Silver issue reason")
if issues.join(day_domain.select(*key_columns), key_columns, "left_anti").count() != 0:
    raise ValueError("Orphan issue vehicle-day")
issues.orderBy("event_date", "vehicle_id", "primary_priority").show(truncate=False)

## 10. Count issues and choose the primary issue

Count all distinct issues while selecting the lowest-priority number as each day's primary explanation.

In [ ]:
issue_counts = issues.groupBy(*key_columns).agg(F.count("*").alias("issue_count"))
primary_priorities = issues.groupBy(*key_columns).agg(F.min("primary_priority").alias("primary_priority"))
primary_issues = (primary_priorities.join(issue_dictionary, "primary_priority", "left")
    .select(*key_columns, F.col("issue_code").alias("primary_issue")))
quality = (quality.join(issue_counts, key_columns, "left")
    .join(primary_issues, key_columns, "left")
    .withColumn("issue_count", F.coalesce("issue_count", F.lit(0))))
primary_issues.orderBy("event_date", "vehicle_id").show(truncate=False)

## 11. Build the shared vehicle-day key

Use the same readable key in both day-level outputs so each issue links to its reporting day.

### Learning note: `F.concat_ws`
`F.concat_ws("|", "vehicle_id", "event_date")` joins the canonical vehicle ID and ISO date with `|`, for example `EV004|2026-08-05`. Both outputs use exactly the same expression. This readable key is not a hash and does not prove uniqueness. Because `concat_ws` skips null values, the natural-key checks run first, and the final checks still verify unique keys. The snapshot reconciliation table has no relationship key.

In [ ]:
quality = quality.withColumn("vehicle_day_key", F.concat_ws("|", "vehicle_id", "event_date"))
issues = issues.withColumn("vehicle_day_key", F.concat_ws("|", "vehicle_id", "event_date"))
quality = quality.select("vehicle_day_key", *day_columns, "accepted_reading_count",
    "rejected_record_count", "held_reading_count", "issue_count", "primary_issue")
issues = issues.select("vehicle_day_key", *key_columns, "issue_code", "issue_label", "primary_priority")

## 12. Reconcile the whole saved snapshot

Account for record outcomes across the snapshot while treating the duplicate remainder as a residual.

In [ ]:
bronze_record_count = bronze.count()
accepted_record_count = accepted.count()
rejected_record_count = rejected.count()
held_record_count = held.count()
quarantine_record_count = quarantine.count()
unlinked_reject_count = rejected.join(day_domain.select(*key_columns), key_columns, "left_anti").count()
linked_reject_count = rejected_record_count - unlinked_reject_count
# Remainder (expected removed duplicates): a residual, not independent proof of deduplication.
removed_duplicate_remainder = bronze_record_count - accepted_record_count - rejected_record_count - held_record_count
reconciliation = spark.createDataFrame([
    (bronze_record_count, accepted_record_count, rejected_record_count, held_record_count,
     quarantine_record_count, linked_reject_count, unlinked_reject_count, removed_duplicate_remainder)
], "bronze_record_count long, accepted_record_count long, rejected_record_count long, "
   "held_record_count long, quarantine_record_count long, linked_reject_count long, "
   "unlinked_reject_count long, removed_duplicate_remainder long")
reconciliation.show(truncate=False)

## 13. Check source scopes, coverage and explanations

Stop publication when source records or untrusted days cannot be linked to a valid reporting explanation.

In [ ]:
if quarantine.filter(F.col("quarantine_scope").isNull() |
        ~F.col("quarantine_scope").isin("record", "day_hold")).count() != 0:
    raise ValueError("Unsupported quarantine scope")
if accepted.join(day_domain.select(*key_columns), key_columns, "left_anti").count() != 0:
    raise ValueError("Orphan accepted vehicle-day")
if held.join(day_domain.select(*key_columns), key_columns, "left_anti").count() != 0:
    raise ValueError("Orphan held vehicle-day")
if day_domain.filter(F.col("distance_status") == "untrusted").join(
        silver_issues.select(*key_columns), key_columns, "left_anti").count() != 0:
    raise ValueError("Untrusted day missing Silver reason")

## 14. Check output grain, links and count identities

Confirm the outputs retain their intended grain and reconcile before replacing the saved tables.

In [ ]:
if quality.count() != day_domain.count():
    raise ValueError("Quality joins changed the day domain")
if quality.groupBy("vehicle_day_key").count().filter(F.col("count") > 1).count() != 0:
    raise ValueError("Duplicate quality key")
if issues.groupBy("vehicle_day_key", "issue_code").count().filter(F.col("count") > 1).count() != 0:
    raise ValueError("Duplicate issue key")
if issues.join(quality.select("vehicle_day_key"), "vehicle_day_key", "left_anti").count() != 0:
    raise ValueError("Orphan issue key")
if rejected_record_count + held_record_count != quarantine_record_count:
    raise ValueError("Quarantine scope counts do not match")
if removed_duplicate_remainder < 0:
    raise ValueError("Negative record remainder")

## 15. Save the three quality outputs

Persist the validated tables so reporting can refresh from the completed quality snapshot.

In [ ]:
# Separate Delta overwrites are not an atomic snapshot. Refresh only after all writes succeed.
quality.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".gold_vehicle_day_quality")
issues.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".gold_vehicle_day_issue")
reconciliation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(schema_name + ".gold_record_reconciliation")
print("Quality days:", quality.count(), "Issue occurrences:", issues.count(), "Snapshot rows:", reconciliation.count())